In [2]:
import sqlite3
import requests

# 1. Pobierz dane z API
response = requests.get("https://randomuser.me/api/?results=30")
users = response.json()["results"]

# 2. Stwórz tabelę Users (id, first_name, last_name, email, age, gender, country)
conn = sqlite3.connect("users.db")
cursor = conn.cursor()

# Usunięcie tabeli, jeśli istnieje, w celu resetowania danych
cursor.execute("DROP TABLE IF EXISTS Users")

cursor.execute("""
CREATE TABLE Users (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    first_name TEXT,
    last_name TEXT,
    email TEXT,
    age INTEGER,
    gender TEXT,
    country TEXT
)
""")
conn.commit()

# 3. Wstaw dane z parametryzacją (? nie f-string!)
for user in users:
    first_name = user["name"]["first"]
    last_name = user["name"]["last"]
    email = user["email"]
    age = user["dob"]["age"]
    gender = user["gender"]
    country = user["location"]["country"]
    
    cursor.execute("""
    INSERT INTO Users (first_name, last_name, email, age, gender, country)
    VALUES (?, ?, ?, ?, ?, ?)
    """, (first_name, last_name, email, age, gender, country))

conn.commit()

# 4. Zapytania analityczne
print("--- Rozkład płci (SELECT gender, COUNT(*) FROM Users GROUP BY gender) ---")
cursor.execute("SELECT gender, COUNT(*) FROM Users GROUP BY gender")
for row in cursor.fetchall():
    print(f"Płeć: {row[0]}, Liczba: {row[1]}")

print("\n--- Średni wiek (SELECT AVG(age) FROM Users) ---")
cursor.execute("SELECT AVG(age) FROM Users")
avg_age = cursor.fetchone()[0]
print(f"Średni wiek użytkowników: {avg_age:.2f} lat")

print("\n--- Kraje zamieszkania (SELECT country, COUNT(*) FROM Users GROUP BY country ORDER BY COUNT(*) DESC) ---")
cursor.execute("SELECT country, COUNT(*) FROM Users GROUP BY country ORDER BY COUNT(*) DESC")
rows = cursor.fetchall()
print(f"Użytkownicy mieszkają w {len(rows)} różnych krajach:")
for row in rows:
    print(f" - {row[0]}: {row[1]} os.")

# Zamknięcie połączenia
conn.close()

--- Rozkład płci (SELECT gender, COUNT(*) FROM Users GROUP BY gender) ---
Płeć: female, Liczba: 12
Płeć: male, Liczba: 18

--- Średni wiek (SELECT AVG(age) FROM Users) ---
Średni wiek użytkowników: 51.93 lat

--- Kraje zamieszkania (SELECT country, COUNT(*) FROM Users GROUP BY country ORDER BY COUNT(*) DESC) ---
Użytkownicy mieszkają w 16 różnych krajach:
 - Germany: 5 os.
 - France: 4 os.
 - Ireland: 3 os.
 - Finland: 3 os.
 - Spain: 2 os.
 - Serbia: 2 os.
 - Netherlands: 2 os.
 - United States: 1 os.
 - United Kingdom: 1 os.
 - Turkey: 1 os.
 - New Zealand: 1 os.
 - Mexico: 1 os.
 - Iran: 1 os.
 - Denmark: 1 os.
 - Canada: 1 os.
 - Australia: 1 os.
